# 2 Ensamblados paralelos heterogéneos 

Para esta parte utilizaremos un famoso dataset sobre la calidad de vinos, que se puede descargar desde https://archive.ics.uci.edu/dataset/186/wine+qualitym

La única diferencia con los ensamblados paralelos homogéneos es que en este caso los modelos base pueden ser de cualquier tipo. Por ejemplo en un problema de clasificación podemos armar un ensamblado heterogéneo con un modelo de regresión logística, un modelo de vecino más cercano, una red neuronal etc. 

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score ,classification_report
# modelos base
from sklearn.linear_model import LogisticRegression
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier
# modelo meta
from sklearn.neural_network import MLPClassifier

In [ ]:
# descargar el dataset en tu directorio local y ejecuta el siguiente comando para descomprimirlo en la carpeta data/ensemble_parallel_heterogeneous/
#!unzip wine+quality.zip -d data/ensemble_parallel_heterogeneous/

In [ ]:
df_red_wine = pd.read_csv("data/ensemble_parallel_heterogeneous/winequality-red.csv", sep=';')
df_white_wine = pd.read_csv("data/ensemble_parallel_heterogeneous/winequality-white.csv", sep=';')

In [ ]:
df_white_wine.head()  

In [ ]:
# unimos ambos datasets en uno solo
df_wines = pd.concat([df_red_wine, df_white_wine], ignore_index=True)

In [ ]:
# agrupamos la variable 'quality' en 3 categorias: 'malo' (0-4), 'normal' (5-6) y 'excelente' (7-10)
df_wines['quality_grouped'] = df_wines['quality'].apply(lambda x: 'malo' if x <= 4 else ('normal' if 5 <= x <= 6 else 'excelente'))

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution of the 'quality_grouped' variable
df_wines['quality_grouped'].value_counts().plot(kind='bar', color=['red', 'blue', 'green'])
plt.title('Distribución de la Calidad del Vino Agrupada')
plt.xlabel('Categoría de Calidad')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
X = df_wines.drop(['quality', 'quality_grouped'], axis=1)
Y = df_wines['quality_grouped']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=21)

In [ ]:
modelos_base = [
    ('Logistic Regression', LogisticRegression( random_state=33,class_weight='balanced')),
    ('Gaussian Process', GaussianProcessClassifier(random_state=33)),
    ('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=5)),
    ('Support Vector Machine', SVC(kernel='rbf', probability=True, random_state=33,class_weight='balanced')),
    ('Decision Tree', DecisionTreeClassifier(random_state=33,class_weight='balanced'))
]

In [ ]:
meta_model = MLPClassifier(hidden_layer_sizes=(128,64,32),random_state=33, max_iter=500)

El stacking es un tipo de ensamblado donde se entrena un conjunto de modelos base y se utiliza las predicciones de estos(probabilidad y/o etiqueta) como input para entrenar un meta-modelo (en nuestro caso una red neuronal). en scikit-learn la clase stackinClassifier nos permite realizar esto de forma simple.   

In [ ]:
clf = StackingClassifier(
    estimators=modelos_base, final_estimator=meta_model)


In [ ]:
pipe = make_pipeline(StandardScaler(), clf)

In [ ]:
pipe.fit(X_train, Y_train)

In [ ]:
print(classification_report(Y_test, pipe.predict(X_test)))

In [ ]:
X_train_scaled = pipe.named_steps['standardscaler'].transform(X_train)
X_test_scaled = pipe.named_steps['standardscaler'].transform(X_test)

for nombre, modelo in modelos_base:
    print(f"Entrenando modelo base: {nombre}")
    modelo.fit(X_train_scaled, Y_train)
    y_pred_base = modelo.predict(X_test_scaled)
    print(classification_report(Y_test, y_pred_base)) 

De los resultados podemos concluir , que todos los modelos en general tienen problema a la hora de detectar un vino malo (pocos casos) , el que mejor lo logra es el árbol de decisión, si nos referimos a todos los vinos tenemos que el ensamblado logra superar a los modelos individuales. 